# Peak shaving vs TCIPC analysis

Let's check the results. First we need to copy the results from the cluster to
this PC:
```
rsync -avm --include='*/' --include='*.sql*' --exclude='*' jhummel@login.delftblue.tudelft.nl:../../scratch/jhummel/tip_clearance/data/optimal_tuning/ ./data/optimal_tuning/ --dry-run
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.interpolate import griddata
from weis.visualization.utils import load_OMsql_multi, load_OMsql

plt.style.use("journal.mplstyle")

%matplotlib widget

In [ ]:
# Define the logs to load and give them labels.
logs_to_load = {
    "Free yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_free_yaw.sql",
    "Zero yaw": "../../../data/optimal_tuning/parameter_sweep/ps_vs_ipc_zero_yaw.sql",
}

# Load all datasets
all_data_dicts = {}
for log_name, log_fmt in logs_to_load.items():
    all_data_dicts[log_name] = load_OMsql(log_fmt)
    print(f"Loaded {log_name}: {all_data_dicts[log_name].keys()}")

In [ ]:
# Let's define how we load, scale, and label the data, then make a dataframe.
all_outputs = {
    # ROSCO variables.
    "TCIPC_MaxTipDeflection": {
        "key": "tune_rosco_ivc.TCIPC_MaxTipDeflection",
        "scaling": lambda x: x[0],
        "label": "TCIPC Max Tip Deflection (m)",
    },
    "ps_percent": {
        "key": "tune_rosco_ivc.ps_percent",
        "scaling": lambda x: x[0],
        "label": "Peak Shaving (-)",
    },
    "TCIPC_nHarmonics": {
        "key": "tune_rosco_ivc.TCIPC_nHarmonics",
        "scaling": lambda x: x[0],
        "label": "Number of harmonics",
    },
    "TCIPC_ZeroYawDeflection": {
        "key": "tune_rosco_ivc.TCIPC_ZeroYawDeflection",
        "scaling": lambda x: x[0],
        "label": "Zero yaw deflection",
    },
    # Objectives / responses
    "aep": {
        "key": "aeroelastic.AEP",
        "scaling": lambda x: 1e-6 * x[0],
        "label": "AEP (GWh)",
    },
    "max_TipDxc_towerPassing": {
        "key": "aeroelastic.max_TipDxc_towerPassing",
        "scaling": lambda x: x[0],
        "label": "Max TipDxc Tower Passing (m)",
    },
    "tower_clearance": {
        "key": "aeroelastic.max_TipDxc_towerPassing",
        "scaling": lambda x: 30 - x[0],
        "label": "Tower clearance (m)",
    },
    "TCIPC_amplitude_at_max_deflection": {
        "key": "aeroelastic.TCIPC_amplitude_at_max_deflection",
        "scaling": lambda x: x[0],
        "label": "TCIPC amplitude (deg)",
    },
    "avg_pitch_travel": {
        "key": "aeroelastic.avg_pitch_travel",
        "scaling": lambda x: x[0],
        "label": "Avg Pitch Travel (deg)",
    },
    "DEL_RootMyb": {
        "key": "aeroelastic.DEL_RootMyb",
        "scaling": lambda x: x[0] / 1000,
        "label": "DEL Root Myb (MNm)",
    },
    "max_TwrBsMyt": {
        "key": "aeroelastic.max_TwrBsMyt",
        "scaling": lambda x: x[0] / 1000,
        "label": "Max Tower Base Myt (MNm)",
    },
}

# Build dataframe from mapping for each log
labels = {short: info["label"] for short, info in all_outputs.items()}
all_dfs = []

for log_name, data_dict in all_data_dicts.items():
    df_dict = {}
    for short_label, info in all_outputs.items():
        data = data_dict[info["key"]]
        scaled_data = list(map(info["scaling"], data))
        df_dict[short_label] = scaled_data

    df_temp = pd.DataFrame(df_dict)
    df_temp["log_name"] = log_name  # Add identifier column
    all_dfs.append(df_temp)

# Combine all dataframes
df = pd.concat(all_dfs, ignore_index=True)
print(f"Combined dataframe shape: {df.shape}")
df.head()

## Data exploration

In [ ]:
# Make a plot of the distribution of our design variables.
plt.figure()
sns.scatterplot(
    df,
    x="TCIPC_MaxTipDeflection",
    y="ps_percent",
    style="log_name",
)
plt.show()

In [ ]:
# Plot several outputs/objectives as a function of the design variables.
# Output variables to plot.
outputs = [
    "aep",
    "tower_clearance",
    "avg_pitch_travel",
    "TCIPC_amplitude_at_max_deflection",
    "DEL_RootMyb",
    "max_TwrBsMyt",
]

fig, axs = plt.subplots(len(logs_to_load), len(outputs), figsize=(12, 5))

for i, log_name in enumerate(logs_to_load.keys()):
    for j, output in enumerate(outputs):
        scatter = axs[i, j].scatter(
            df[df["log_name"] == log_name]["ps_percent"],
            df[df["log_name"] == log_name]["TCIPC_MaxTipDeflection"],
            c=df[df["log_name"] == log_name][output],
        )

        axs[i, j].set_xlabel("Peak shaving (-)")
        axs[i, j].set_ylabel("TCIPC max tip deflection (m)")
        axs[i, j].set_title(output)
        plt.colorbar(scatter, ax=axs[i, j])

plt.show()

In [ ]:
# Get an idea of the trade-off between AEP and tower clearance.
fig, ax = plt.subplots()

df.sort_values("aep", inplace=True)

sns.lineplot(
    data=df[df["TCIPC_MaxTipDeflection"] == 0.0],
    x="aep",
    y="tower_clearance",
    hue="log_name",
)
sns.lineplot(
    data=df[df["TCIPC_MaxTipDeflection"] == 20.0],
    x="aep",
    y="tower_clearance",
    hue="log_name",
    palette="deep",
)

## Data interpolation

In [ ]:
# Define the bounds of our design space for interpolation.
ps_min, ps_max = (
    df["ps_percent"].min(),
    df["ps_percent"].max(),
)
tip_min, tip_max = (
    df["TCIPC_MaxTipDeflection"].min(),
    df["TCIPC_MaxTipDeflection"].max(),
)

# Create a regular grid for interpolation to enable smooth contour plots.
n_points = 50
ps_grid = np.linspace(ps_min, ps_max, n_points)
tip_grid = np.linspace(tip_min, tip_max, n_points)
ps_percent_grid, tcipc_reference_grid = np.meshgrid(ps_grid, tip_grid)

# Interpolate each output variable on the grid for each log.
# We store the results in a nested dictionary for easy access when plotting.
interpolated_data = {}

for log_name in logs_to_load.keys():
    interpolated_data[log_name] = {}
    df_log = df[df["log_name"] == log_name]

    # Extract the independent variables as points for interpolation.
    points = df_log[["ps_percent", "TCIPC_MaxTipDeflection"]].values

    for output in outputs:
        # Extract the dependent variable values.
        values = df_log[output].values

        # Interpolate using linear method, which works well for scattered data.
        grid_values = griddata(
            points,
            values,
            (ps_percent_grid, tcipc_reference_grid),
            method="linear",
        )

        interpolated_data[log_name][output] = grid_values

print(f"Interpolated {len(outputs)} outputs for {len(logs_to_load)} datasets")
print(f"Grid shape: {ps_percent_grid.shape}")

In [ ]:
# Plot several outputs/objectives as a function of the design variables.
# Output variables to plot.
outputs = [
    "aep",
    "tower_clearance",
    "avg_pitch_travel",
    "TCIPC_amplitude_at_max_deflection",
    "DEL_RootMyb",
    "max_TwrBsMyt",
]
clims = [
    [70, 95],
    [10, 30],
    [0, 2],
    [0, 5],
    [15, 50],
    [150, 400],
]

fig, axs = plt.subplots(len(logs_to_load), len(outputs), figsize=(12, 5))

for i, log_name in enumerate(logs_to_load.keys()):
    for j, output in enumerate(outputs):
        levels = np.linspace(clims[j][0], clims[j][1], 9)
        scatter = axs[i, j].contourf(
            ps_percent_grid,
            tcipc_reference_grid,
            interpolated_data[log_name][output],
            levels=levels,
            vmin=clims[j][0],
            vmax=clims[j][1],
        )

        axs[i, j].set_xlabel("Peak shaving (-)")
        axs[i, j].set_ylabel("TCIPC max tip deflection (m)")
        axs[i, j].set_title(output)
        plt.colorbar(scatter, ax=axs[i, j])


In [ ]:
# Investigate one plot in detail.
fig, ax = plt.subplots()

scatter = ax.contourf(
    ps_percent_grid,
    tcipc_reference_grid,
    interpolated_data["Free yaw"]["max_TwrBsMyt"],
    # levels=np.linspace(27, 30, 100),
    # levels=np.linspace(0.15, 1.5, 100),
    levels=np.linspace(260, 420, 100),
)

# plt.title("DEL_RootMyb")
plt.colorbar(scatter)

## Optimization

In [ ]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.algorithms.moo.ctaea import CTAEA
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.termination import get_termination
from pymoo.optimize import minimize
from pymoo.indicators.hv import Hypervolume
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator
from copy import deepcopy
from itertools import cycle

In [ ]:
class InterpolatorSet:
    """Builds interpolators for all numeric columns of a dataframe subset.

    Given a dataframe (typically filtered to one log_name), this creates a
    scipy interpolator for each numeric column, using two specified design
    variable columns as inputs. The interpolation method can be "cubic"
    (CloughTocher2D) or "linear" (LinearND).
    """

    DEFAULT_BASELINE = (0.8, 20.0)

    def __init__(
        self,
        df,
        design_vars=("ps_percent", "TCIPC_MaxTipDeflection"),
        method="cubic",
        baseline=None,
    ):
        self.design_vars = list(design_vars)
        self.method = method
        self.baseline = baseline if baseline is not None else self.DEFAULT_BASELINE

        # Extract the interpolation input points.
        xy = df[self.design_vars].values

        # Build an interpolator for every numeric column that is not a
        # design variable.
        self._interpolators = {}
        for col in df.select_dtypes(include=[np.number]).columns:
            if col in self.design_vars:
                continue

            Interpolator = (
                CloughTocher2DInterpolator
                if method == "cubic"
                else LinearNDInterpolator
            )
            self._interpolators[col] = Interpolator(xy, df[col].values)

    def __getitem__(self, column):
        return self._interpolators[column]

    def __contains__(self, column):
        return column in self._interpolators

    @property
    def columns(self):
        return list(self._interpolators.keys())

In [ ]:
class OptimizationProblem:
    """Declarative specification of a multi-objective optimization problem.

    Objectives are specified as (column_name, "minimize" | "maximize") tuples.
    Constraints are callables with signature:
        constraint_factory(interps, baseline_interps) -> callable(x, y) -> float
    where interps is the current dataset's InterpolatorSet and baseline_interps
    is always the baseline dataset's InterpolatorSet (Free yaw). G <= 0 is
    feasible.

    Example constraint (DEL must not exceed baseline + 10% margin):
        lambda interps, bl: lambda x, y: (
            interps["DEL_RootMyb"](x, y)
            - 1.1 * bl["DEL_RootMyb"](*bl.baseline)
        )
    """

    def __init__(
        self,
        objectives,
        constraints=None,
        xl=None,
        xu=None,
        interp_method="cubic",
        design_vars=("ps_percent", "TCIPC_MaxTipDeflection"),
        baseline=None,
    ):
        self.objectives = objectives
        self.constraints = constraints or []
        self.xl = np.array(xl) if xl is not None else np.array([0.5, 0.0])
        self.xu = np.array(xu) if xu is not None else np.array([1.0, 20.0])
        self.interp_method = interp_method
        self.design_vars = design_vars
        self.baseline = baseline

    def make_baseline_variant(self):
        """Create a copy with the second design variable fixed at its upper bound.

        This represents the baseline case where TCIPC is disabled (fixed at
        max tip deflection) and only peak shaving varies.
        """
        variant = deepcopy(self)
        variant.xl = self.xl.copy()
        variant.xl[1] = self.xu[1]
        return variant

    def to_pymoo(self, interp_set, baseline_interp_set):
        """Build a pymoo ElementwiseProblem from this specification.

        The InterpolatorSet provides the actual interpolated data. Objectives
        marked as "maximize" are negated internally so pymoo always minimizes.
        The baseline_interp_set is passed to constraints so they can reference
        baseline values from the Free yaw dataset.
        """
        objectives = self.objectives
        constraints = self.constraints
        xl = self.xl
        xu = self.xu

        # Determine sign for each objective: pymoo always minimizes, so we
        # negate objectives that should be maximized.
        signs = []
        for _, direction in objectives:
            signs.append(-1.0 if direction == "maximize" else 1.0)

        # Bind constraints to both the current and baseline interpolator sets.
        bound_constraints = [c(interp_set, baseline_interp_set) for c in constraints]

        # Build objective callables with the correct sign.
        obj_callables = []
        for (col, _), sign in zip(objectives, signs):
            interp = interp_set[col]
            obj_callables.append(lambda x, y, _i=interp, _s=sign: _s * _i(x, y))

        # Use closures to capture the objective and constraint callables.
        class _Problem(ElementwiseProblem):
            def __init__(self):  # noqa: N805
                super().__init__(
                    n_var=2,
                    n_obj=len(objectives),
                    n_ieq_constr=len(bound_constraints),
                    xl=xl,
                    xu=xu,
                )

            def _evaluate(self, x, out, *args, **kwargs):  # noqa: N805
                out["F"] = [f(*x) for f in obj_callables]
                out["G"] = [g(*x) for g in bound_constraints]

        return _Problem()

    @property
    def objective_labels(self):
        return [labels.get(col, col) for col, _ in self.objectives]

    @property
    def objective_signs(self):
        """Return the sign array (for converting pymoo output back)."""
        return [-1.0 if d == "maximize" else 1.0 for _, d in self.objectives]

In [ ]:
class OptimizationStudy:
    """Orchestrates multi-objective optimization across datasets and algorithms.

    Runs the same OptimizationProblem on every log_name in the dataframe,
    using every algorithm in the provided dict. Also runs a baseline variant
    (second design variable fixed) using a single specified dataset.
    Constraints always reference the baseline dataset's interpolators so that
    constraint bounds are consistent across all datasets.
    Provides methods to compare convergence, Pareto fronts, and design spaces.
    """

    MARKERS = ["o", "s", "^", "D", "v", "P", "*", "X"]

    def __init__(
        self,
        df,
        problem,
        algorithms,
        termination,
        baseline_log="Free yaw",
        log_names=None,
        seed=1,
        verbose=False,
    ):
        self.df = df
        self.problem = problem
        self.algorithms = algorithms
        self.termination = termination
        self.baseline_log = baseline_log
        self.log_names = log_names or df["log_name"].unique().tolist()
        self.seed = seed
        self.verbose = verbose

        # Populated by run().
        self.results = {}
        self.interp_sets = {}

    def run(self):
        """Run all optimizations: each (log_name, algorithm) combination.

        The baseline variant (second design variable fixed at its upper bound)
        is only run for baseline_log, since the baseline represents the turbine
        without TCIPC under the Free yaw condition. Constraints always
        evaluate against the baseline dataset's interpolators.
        """
        # Build interpolator sets for all datasets first.
        for log_name in self.log_names:
            df_log = self.df[self.df["log_name"] == log_name]
            self.interp_sets[log_name] = InterpolatorSet(
                df_log,
                design_vars=self.problem.design_vars,
                method=self.problem.interp_method,
                baseline=self.problem.baseline,
            )

        # The baseline interpolator set is used for constraint evaluation
        # across all datasets, ensuring consistent constraint bounds.
        bl_interp = self.interp_sets[self.baseline_log]

        # Run baseline variant once using the designated baseline dataset.
        if self.baseline_log and self.baseline_log in self.interp_sets:
            baseline_problem = self.problem.make_baseline_variant()
            pymoo_baseline = baseline_problem.to_pymoo(bl_interp, bl_interp)

            for algo_name, algo_factory in self.algorithms.items():
                algo_bl = deepcopy(algo_factory)
                result_bl = minimize(
                    pymoo_baseline,
                    algo_bl,
                    deepcopy(self.termination),
                    seed=self.seed,
                    save_history=True,
                    verbose=self.verbose,
                )
                bl_key = ("Baseline", algo_name)
                self.results[bl_key] = result_bl
                print(f"  Done: Baseline / {algo_name} -> {len(result_bl.F)} solutions")

        # Run full-range optimizations for each dataset and algorithm.
        for log_name in self.log_names:
            interp_set = self.interp_sets[log_name]

            for algo_name, algo_factory in self.algorithms.items():
                pymoo_problem = self.problem.to_pymoo(interp_set, bl_interp)
                algo = deepcopy(algo_factory)
                result = minimize(
                    pymoo_problem,
                    algo,
                    deepcopy(self.termination),
                    seed=self.seed,
                    save_history=True,
                    verbose=self.verbose,
                )
                key = (log_name, algo_name)
                self.results[key] = result
                print(f"  Done: {log_name} / {algo_name} -> {len(result.F)} solutions")

    def _iter_results(self):
        """Yield (label, result) tuples in a consistent order."""
        for key in self.results:
            log_name, algo_name = key
            yield f"{log_name} / {algo_name}", self.results[key]

    @staticmethod
    def _calculate_hv_history(result):
        """Compute hypervolume at each generation from a pymoo result."""
        hist_F = []
        for algo in result.history:
            opt = algo.opt
            feas = np.where(opt.get("feasible"))[0]
            hist_F.append(opt.get("F")[feas])

        approx_ideal = result.F.min(axis=0)
        approx_nadir = result.F.max(axis=0)

        metric = Hypervolume(
            ref_point=approx_nadir,
            norm_ref_point=False,
            zero_to_one=False,
            ideal=approx_ideal,
            nadir=approx_nadir,
        )
        return [metric.do(_F) for _F in hist_F]

    def plot_convergence(self, ax=None):
        """Plot hypervolume convergence for all optimization runs."""
        if ax is None:
            fig, ax = plt.subplots()
        markers = cycle(self.MARKERS)

        for label, result in self._iter_results():
            hv = self._calculate_hv_history(result)
            ax.plot(hv, marker=next(markers), lw=1.5, markersize=3, label=label)

        ax.set_xlabel("Generation")
        ax.set_ylabel("Hypervolume")
        ax.legend()
        return ax

    def plot_pareto(self, ax=None):
        """Plot Pareto front comparison for all runs.

        For 2 objectives: 2D line plot. For 3 objectives: 3D scatter plot.
        Objective values are converted back to the original direction
        (un-negating maximized objectives).
        """
        n_obj = len(self.problem.objectives)
        signs = np.array(self.problem.objective_signs)
        obj_labels = self.problem.objective_labels

        if n_obj == 2:
            if ax is None:
                fig, ax = plt.subplots()
            markers = cycle(self.MARKERS)

            for label, result in self._iter_results():
                # Convert back from pymoo's minimization convention.
                F = result.F * signs
                sort_idx = F[:, 0].argsort()
                F = F[sort_idx]
                ax.plot(
                    F[:, 0],
                    F[:, 1],
                    marker=next(markers),
                    label=label,
                    markersize=4,
                )

            ax.set_xlabel(obj_labels[0])
            ax.set_ylabel(obj_labels[1])
            ax.legend()

        elif n_obj == 3:
            if ax is None:
                fig = plt.figure()
                ax = fig.add_subplot(111, projection="3d")
            markers = cycle(self.MARKERS)

            for label, result in self._iter_results():
                F = result.F * signs
                ax.scatter(
                    F[:, 0],
                    F[:, 1],
                    F[:, 2],
                    marker=next(markers),
                    label=label,
                    s=15,
                )

            ax.set_xlabel(obj_labels[0])
            ax.set_ylabel(obj_labels[1])
            ax.set_zlabel(obj_labels[2])
            ax.legend()

        return ax

    def plot_design_space(self, ax=None):
        """Plot design variable values for all Pareto-optimal points."""
        if ax is None:
            fig, ax = plt.subplots()
        markers = cycle(self.MARKERS)

        for label, result in self._iter_results():
            ax.scatter(
                result.X[:, 0],
                result.X[:, 1],
                marker=next(markers),
                s=20,
                label=label,
            )

        ax.set_xlim(self.problem.xl[0], self.problem.xu[0])
        ax.set_ylim(0, self.problem.xu[1])
        ax.set_xlabel(
            labels.get(self.problem.design_vars[0], self.problem.design_vars[0])
        )
        ax.set_ylabel(
            labels.get(self.problem.design_vars[1], self.problem.design_vars[1])
        )
        ax.legend()
        return ax

In [ ]:
# Define the optimization problem declaratively.
problem = OptimizationProblem(
    objectives=[
        ("aep", "maximize"),
        ("tower_clearance", "maximize"),
    ],
    constraints=[
        # Blade root DEL must not exceed the baseline value.
        lambda interps, bl: (
            lambda x, y: (
                interps["DEL_RootMyb"](x, y) - 1.0 * bl["DEL_RootMyb"](*bl.baseline)
            )
        ),
        lambda interps, bl: (
            lambda x, y: (
                interps["max_TwrBsMyt"](x, y) - 1.0 * bl["max_TwrBsMyt"](*bl.baseline)
            )
        ),
    ],
    interp_method="linear",
)

# Specify the algorithms to compare. Each entry gets a fresh copy per run.
ref_dirs = get_reference_directions("uniform", 2, n_partitions=100)

algorithms = {
    # "NSGA2": NSGA2(
    #     pop_size=200,
    #     n_offsprings=20,
    #     sampling=FloatRandomSampling(),
    #     crossover=SBX(prob=0.9, eta=15),
    #     mutation=PM(eta=15),
    #     eliminate_duplicates=True,
    # ),
    # MOEAD cannot do constraints?
    # "MOEAD": MOEAD(
    #     ref_dirs,
    #     n_neighbors=50,
    #     prob_neighbor_mating=0.7,
    # ),
    "CTAEA": CTAEA(
        ref_dirs,
        sampling=FloatRandomSampling(),
        crossover=SBX(prob=0.9, eta=5),
        mutation=PM(eta=10),
        eliminate_duplicates=True,
    ),
}

termination = get_termination("n_gen", 200)

# Run the study across all datasets and algorithms.
study = OptimizationStudy(
    df,
    problem=problem,
    algorithms=algorithms,
    termination=termination,
    verbose=False,
)
study.run()

In [ ]:
# Hypervolume convergence comparison across all runs.
study.plot_convergence()
plt.show()

In [ ]:
# Pareto front comparison across all runs.
study.plot_pareto()
plt.show()

In [ ]:
# Design space comparison across all runs.
study.plot_design_space()
plt.show()